# Подключение к гугл диску

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Загрузка библиотек и данных

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import TimeDistributed, SpatialDropout1D, Bidirectional, Lambda

pd.options.plotting.backend = "plotly"

tracks_logs = pd.read_csv(r'/content/test.csv', sep=';')
tracks_logs

,Проект,Дата,Кол-во минут,Номер тикета,Вид работ,Заметка,Оплачено,Суммарное время (минуты)
0,Devstars,2023-08-11 00:00:00,30,0,Documentation,создание отчета,0,30
1,Devstars,2023-08-11 00:00:00,70,0,Communication,обсуждение трекера и отчетов,0,70
2,Кладовочка,2023-08-11 00:00:00,5,0,Communication,планерка,0,5
3,Пкаско,2023-08-11 00:00:00,33,0,Communication,созвон по Пкаско,0,33
4,ПКаско(Согласие),2023-08-11 00:00:00,35,0,Communication,подготовка к оценке проекта,0,35
...,...,...,...,...,...,...,...,...
10900,Пкаско,2022-06-11 00:00:00,15,0,Communication,обсуждение mvp,0,15
10901,Пкаско,2022-06-11 00:00:00,34,0,Communication,декомпозиция mvp,0,34
10902,Пкаско,2022-06-11 00:00:00,30,0,Communication,декомпозиция mvp,0,30
10903,Пкаско,2022-06-11 00:00:00,15,0,Communication,stand up,0,15


In [ ]:
tf.config.run_functions_eagerly(True)

text_vec = tf.keras.layers.TextVectorization(output_sequence_length=25,
                                             standardize="lower_and_strip_punctuation")

# Генерация данных

In [ ]:
months = {1: 'Январь',
          2: 'Февраль',
          3: 'Март',
          4: 'Апрель',
          5: 'Май',
          6: 'Июнь',
          7: 'Июль',
          8: 'Август',
          9: 'Сентябрь',
          10: 'Октябрь',
          11: 'Ноябрь',
          12: 'Декабрь'}

In [ ]:
def random_concatenate(row_):
    """
    Объединение текстовых данных
    :param row: строка данных
    :return row: предподготовленная строка
    """
    row = row_.copy()
    # Форматы времени на работу
    minutesv1 = f"{row['Кол-во минут']} минут"
    minutesv2 = f"{row['Кол-во минут']}"
    # Виды тикетов
    if row['Номер тикета'] == 0:
        row['Номер тикета'] = np.random.randint(0, 20000)

    ticket_v2 = f"{row['Номер тикета']} тикет"
    ticket_v3 = f"тикет {row['Номер тикета']}"
    # Дни недели
    days = ['Сегодня', 'Вчера', 'Понедельник', 'Вторник',
            'Среда', 'Четверг', 'Пятница']
    date = np.random.choice([pd.to_datetime(row['Дата']).strftime('%d-%m'),
                             pd.to_datetime(row['Дата']).strftime('%d.%m.%y'),
                             pd.to_datetime(row['Дата']).strftime('%d.%m')])

    month = int(row['Дата'].split('-')[1])
    if np.random.randint(0, 2) == 0 and month in [1, 2, 4, 5, 6, 7, 9, 10, 11, 12]:
        month = months[month]
        month = month[:-1] + 'я'
    elif np.random.randint(0, 2) == 0 and month in [3, 8]:
        month = months[month]
        month = month[:-1] + 'а'
    else:
        month = months[month]

    _day = row['Дата'].split('-')[-1].split(' ')[0]

    day = np.random.choice(days)
    random_date = np.random.randint(0, 3)

    row['Дата'] = day

    date2 = str(int(_day)) + ' ' + month
    # Шаблоны тегов
    tags_v1 = ['B-PRO', 'B-DAT', ['B-MIN', 'I-MIN'], ['B-TKT', 'I-TKT'], [*np.full(len(row['Заметка'].split(' ')), '0')]]
    tags_v2 = ['B-PRO', ['B-DAT', 'I-DAT'], ['B-MIN', 'I-MIN'], ['B-TKT', 'I-TKT'], [*np.full(len(row['Заметка'].split(' ')), '0')]]
    tags_v3 = ['B-PRO', ['B-DAT', 'I-DAT'], ['B-MIN', 'I-MIN'], ['B-TKT', 'I-TKT'], [*np.full(len(row['Заметка'].split(' ')), '0')]]
    tags_v4 = ['B-PRO', ['B-DAT', 'I-DAT'], 'B-MIN', [*np.full(len(row['Заметка'].split(' ')), '0')]]

    type_work = row['Вид работ']
    row = row[['Проект', 'Дата', 'Кол-во минут', 'Номер тикета', 'Заметка']]

    tags_type = np.random.randint(0, 5)
    # Перемешивание
    x = np.random.randint(3, 6)
    rng = np.random.default_rng()
    rnd_indexes = rng.choice(list(range(5)), size=x, replace=False)

    if tags_type == 0:
        row['Кол-во минут'] = minutesv1
        if np.random.randint(0, 2) == 1:
            row['Номер тикета'] = ticket_v2
        else:
            row['Номер тикета'] = ticket_v3
        concanated_row = ' '.join(str(row.iloc[x]) for x in rnd_indexes)
        tags = [[tags_v1[x]] for x in rnd_indexes]
        row['Код'] = 1
    elif tags_type == 1:
        row['Кол-во минут'] = minutesv1
        if np.random.randint(0, 2) == 1:
            row['Номер тикета'] = ticket_v2
        else:
            row['Номер тикета'] = ticket_v3
        concanated_row = ' '.join(str(row.iloc[x]) for x in rnd_indexes)
        tags = [[tags_v1[x]for x in rnd_indexes]]
        row['Код'] = 2
    elif tags_type == 2:
        row['Кол-во минут'] = minutesv1
        if np.random.randint(0, 2) == 1:
            row['Номер тикета'] = ticket_v2
        else:
            row['Номер тикета'] = ticket_v3
        row['Дата'] = date2
        concanated_row = ' '.join(str(row.iloc[x]) for x in rnd_indexes)
        tags = [[tags_v2[x]] for x in rnd_indexes]
        row['Код'] = 3
    elif tags_type == 3:
        row['Кол-во минут'] = minutesv1
        row['Номер тикета'] = ticket_v2
        row['Дата'] = date2
        concanated_row = ' '.join(str(row.iloc[x]) for x in rnd_indexes)
        tags = [[tags_v3[x]] for x in rnd_indexes]
        row['Код'] = 4
    elif tags_type == 4:
        x = np.random.randint(3, 5)
        rng = np.random.default_rng()
        rnd_indexes = rng.choice(list(range(4)), size=x, replace=False)
        row['Кол-во минут'] = minutesv2
        row = row.drop('Номер тикета')
        row['Дата'] = date2
        concanated_row = ' '.join(str(row.iloc[x]) for x in rnd_indexes)
        tags = [[tags_v4[x]] for x in rnd_indexes]
        row['Код'] = 5

    flatten_list  = lambda tags: [element for item in tags for element in flatten_list(item)] if type(tags) is list else [tags]

    row['Вид работ'] = type_work
    row['Заметка'] = concanated_row
    row['Tag'] = flatten_list(tags)

    if '0' not in row['Tag']:
        row['Вид работ'] = 'Without text info'
    return row

In [ ]:
test_samples = tracks_logs.sample(400)
train_samples = tracks_logs[~tracks_logs.index.isin(test_samples.index)]

In [ ]:
%%time

tracks_logs['Tag'] = 0
tracks_logs['Код'] = -1
logs = train_samples.copy()
train_tracks_logs_copy = train_samples.copy()

for iter in range(70):
    copy = logs.copy()
    copy = copy.apply(random_concatenate, axis=1)
    train_tracks_logs_copy = train_tracks_logs_copy.append(copy,
                                                           ignore_index=True)

CPU times: user 40.9 s, sys: 363 ms, total: 41.2 s
Wall time: 49.3 s


In [ ]:
logs = test_samples.copy()
test_tracks_logs_copy = test_samples.copy()

for iter in range(70):
    copy = logs.copy()
    copy = copy.apply(random_concatenate, axis=1)
    test_tracks_logs_copy = test_tracks_logs_copy.append(copy,
                                                         ignore_index=True)

In [ ]:
train_tracks_logs_copy = train_tracks_logs_copy[train_tracks_logs_copy['Tag'] != 0]
train_tracks_logs_copy = train_tracks_logs_copy.drop_duplicates('Заметка')

In [ ]:
train_tracks_logs_copy = train_tracks_logs_copy.dropna(subset=['Tag'])

In [ ]:
test_tracks_logs_copy = test_tracks_logs_copy[test_tracks_logs_copy['Tag'] != 0]
test_tracks_logs_copy = test_tracks_logs_copy.drop_duplicates('Заметка')
test_tracks_logs_copy = test_tracks_logs_copy.dropna(subset=['Tag'])

## Вставка дополнительного тега для сложных проектов

In [ ]:
difficult_projects = []
for project in tracks_logs['Проект'].unique():
    if ' ' in project:
        difficult_projects.append(project)

In [ ]:
for project in difficult_projects:
    for _, row in train_tracks_logs_copy[train_tracks_logs_copy['Проект'] == project].iterrows():
        if 'B-PRO' in row['Tag']:
            row['Tag'].insert(row['Tag'].index('B-PRO')+1, 'I-PRO')

In [ ]:
for project in difficult_projects:
    for _, row in test_tracks_logs_copy[test_tracks_logs_copy['Проект'] == project].iterrows():
        if 'B-PRO' in row['Tag']:
            row['Tag'].insert(row['Tag'].index('B-PRO')+1, 'I-PRO')

## Вставка года

In [ ]:
def add_year(row):
    if 'I-DAT' in row['Tag']:
        if np.random.randint(0, 2) == 1:
            index = row['Tag'].index('I-DAT') + 1
            row['Tag'].insert(index, 'I-DAT')
            text = row['Заметка'].split()
            text.insert(index, str(np.random.randint(2022, 2025)))
            row['Заметка'] = ' '.join(text)
            row['Код'] = 100
    return row

train_tracks_logs_copy = train_tracks_logs_copy.apply(add_year, axis=1)

## Замена слов

In [ ]:
# Ключ - старое слово, значение - новое слово
words = {
    'анбординг': 'анбоардинг',
    'дашборд': 'дэшборд',
    'стенд ап': 'станд ап'
}

def words_changer(row):
    if np.random.randint(0, 2) == 0:
        for old, new in words.items():
            row['Заметка'] = row['Заметка'].lower().replace(old, new)
    return row

train_tracks_logs_copy = train_tracks_logs_copy.apply(words_changer, axis=1)

# Подготовка данных для модели

In [ ]:
tag2int = {'O': 9,
           '0': 0,
           'B-DAT': 1,
           'I-DAT': 2,
           'B-MIN': 3,
           'I-MIN': 4,
           'B-TKT': 5,
           'I-TKT': 6,
           'B-PRO': 7,
           'I-PRO': 8
           }

In [ ]:
MAX_LEN = 25

In [ ]:
y_train = [[tag2int[tag] for tag in x] for x in train_tracks_logs_copy['Tag']]
y_train = pad_sequences(maxlen=MAX_LEN,
                        sequences=y_train,
                        padding='post',
                        value=tag2int["O"])

In [ ]:
y_train = [to_categorical(i, num_classes=len(tag2int)) for i in y_train]

In [ ]:
y_test = [[tag2int[tag] for tag in x] for x in test_tracks_logs_copy['Tag']]
y_test = pad_sequences(maxlen=MAX_LEN,
                       sequences=y_test,
                       padding='post',
                       value=tag2int["O"])

In [ ]:
y_test = [to_categorical(i, num_classes=len(tag2int)) for i in y_test]

In [ ]:
le = LabelEncoder()
y_train2 = le.fit_transform(train_tracks_logs_copy['Вид работ'])

In [ ]:
y_train2 = to_categorical(y_train2)

In [ ]:
y_test2 = to_categorical(le.transform(test_tracks_logs_copy['Вид работ']))

In [ ]:
train_tracks_logs_copy_lower = train_tracks_logs_copy['Заметка'].map(lambda x: x.lower())
test_tracks_logs_copy_lower = test_tracks_logs_copy['Заметка'].map(lambda x: x.lower())

In [ ]:
X_train = train_tracks_logs_copy_lower
X_test = test_tracks_logs_copy_lower

In [ ]:
X_train_ds = tf.data.Dataset.from_tensor_slices(train_tracks_logs_copy_lower.values.reshape(-1, 1))
text_vec.adapt(X_train_ds.batch(64))

In [ ]:
input_word = Input(shape=(25))
model = Embedding(input_dim=text_vec.vocabulary_size(),
                  output_dim=128,
                  input_length=MAX_LEN)(input_word)
model = Bidirectional(LSTM(units=128, return_sequences=True))(model)
model = Bidirectional(LSTM(units=64, return_sequences=True))(model)

out1 = TimeDistributed(Dense(10, activation='softmax'), name='ner')(model)
out2 = tf.keras.layers.Flatten()(model)
out2 = Dense(64, activation='relu')(out2)
out2 = Dense(9, activation='softmax', name='classification')(out2)

model = Model(input_word, [out1, out2])
model.summary()

In [ ]:
losses = {
    'ner': 'categorical_crossentropy',
    'classification': 'categorical_crossentropy'
    }

trainTargets = {
    'ner': np.array(y_train),
    'classification': np.array(y_train2)
    }

validationTargets = {
    'ner': np.array(y_test),
    'classification': np.array(y_test2)
    }

In [ ]:
model.compile(optimizer='adam',
              loss=losses,
              metrics=['accuracy'])

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_ner_accuracy',
                                              verbose=1,
                                              min_delta=0.2,
                                              patience=3,
                                              restore_best_weights=True)]

In [ ]:
history = model.fit(
    text_vec(X_train), trainTargets,
    validation_data=(text_vec(X_test), validationTargets),
    batch_size=128,
    epochs=2,
    verbose=1)

# Проверка

In [ ]:
tags = {value: key for key, value in tag2int.items()}
STR = '60 минут работа над грантом'.lower()

p = model.predict(text_vec([STR]))
p = np.argmax(p[0], axis=-1)

print("{:15}{:5}\t{}\n".format("Word", "True", "Pred"))
print("-"*30)

for (w, pred) in zip(STR.split(' '), p[0]):
    print("{:15}\t{}".format(w,  tags[pred]))

# Сохранение результатов

## Сохранение модели для последующего преобразования

In [ ]:
class Export(tf.Module):
    def __init__(self, model):
        self.model = model

    @tf.function(input_signature=[tf.TensorSpec(dtype=tf.float32, shape=(None, 25))])
    def convert(self, inputs):
        return self.model(inputs)


export = Export(model)

tf.saved_model.save(export, 'model',
                    signatures={'serving_default': export.convert})

In [ ]:
%cd drive/MyDrive
%mkdir model
%cd model

## Сохранение словаря

In [ ]:
import pickle

config = text_vec.get_config()
try:
  config.pop('encoding')
except KeyError:
  pass

pickle.dumps(pickle.dump({'config': text_vec.get_config(),
                          'weights': text_vec.get_weights()},
             open(r"/content/drive/MyDrive/model/tv_layer.pkl", "wb")))

## Сохранение модели onnx

In [ ]:
!pip install tf2onnx

In [ ]:
!python -m tf2onnx.convert --saved-model /content/model --output /content/drive/MyDrive/model/model.onnx

In [ ]:
tf.keras.models.save_model(model, '/content/drive/MyDrive/model')